# ContextEditingMiddleware
## 举例1，实验组，启用上下文编辑

In [2]:
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 全局计数器，用于在工具内部追踪这是第几次被触发
count = 0


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count

    # 故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
            f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
            f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
            f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
            f"可以出门玩，可以跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            # 配置“清除工具调用记录”策略
            # 作用：当上下文中的历史消息累积满足特定条件时，自动裁剪/删除旧的工具调用及返回结果
            edits=[
                ClearToolUsesEdit(
                    trigger=50,# 触发阈值（例如：当工具返回的 Token 数或消息数达到设定值时触发，具体取决于LangChain版本实现）
                    keep=0 # 触发清理时，保留最近的几次工具调用。这里设置为 0 代表全部清除旧的工具消息
                )
            ]
        )
    ],
    # 内存检查点管理器：用于保存多轮对话的上下文（实现多轮对话记忆）
    checkpointer=InMemorySaver(),

)

# 配置项：定义会话的 thread_id。相同的 id 代表同一个用户的连续对话
config = {"configurable":{"thread_id":"1"}}
# 模拟连续 3 轮的用户多轮对话
for i in range(3):
     print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
     count +=1
     # 调用智能体，传入当前轮次的用户提问，并携带历史会话配置 config
     resp = agent.invoke({"messages":[HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},config=config)
     for msg in resp['messages']:
         # msg.pretty_print()
         if isinstance(msg,AIMessage):
             if not msg.tool_calls:
                 print(f"本次token用量：{msg.usage_metadata}")




============================== 当前是第 1 轮调用 ==============================
本次token用量：{'input_tokens': 105, 'output_tokens': 8, 'total_tokens': 113, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
============================== 当前是第 2 轮调用 ==============================
本次token用量：{'input_tokens': 105, 'output_tokens': 8, 'total_tokens': 113, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
本次token用量：{'input_tokens': 132, 'output_tokens': 92, 'total_tokens': 224, 'input_token_details': {'cache_read': 100}, 'output_token_details': {}}
============================== 当前是第 3 轮调用 ==============================
本次token用量：{'input_tokens': 105, 'output_tokens': 8, 'total_tokens': 113, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
本次token用量：{'input_tokens': 132, 'output_tokens': 92, 'total_tokens': 224, 'input_token_details': {'cache_read': 100}, 'output_token_details': {}}
本次token用量：{'input_tokens': 162, 'output_tokens': 146, 'to

## 举例2，对照组，不裁剪上下文

In [3]:
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver

# 1.初始化模型
model = ChatOpenAI(
    # model="qwen3-vl:latest",
    # model="483025889/qwen3.5:9b",
    model="gemma4:12b",
    profile={"max_input_tokens": 128_000},
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 全局计数器，用于在工具内部追踪这是第几次被触发
count = 0


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count

    # 故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
            f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
            f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
            f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
            f"可以出门玩，可以跑步，钓鱼，爬山，一切都很好哈哈哈")

# 作为对照，这里不使用中间件
agent = create_agent(
    model=model,
    tools=[get_weather],
    # 内存检查点管理器：用于保存多轮对话的上下文（实现多轮对话记忆）
    checkpointer=InMemorySaver(),

)

# 配置项：定义会话的 thread_id。相同的 id 代表同一个用户的连续对话
config = {"configurable":{"thread_id":"1"}}
# 模拟连续 3 轮的用户多轮对话
for i in range(3):
     print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
     count +=1
     # 调用智能体，传入当前轮次的用户提问，并携带历史会话配置 config
     resp = agent.invoke(
         {"messages":[HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},
         config=config
     )
     for msg in resp['messages']:
         # msg.pretty_print()
         if isinstance(msg,AIMessage):
             if not msg.tool_calls:
                 print(f"本次token用量：{msg.usage_metadata}")




============================== 当前是第 1 轮调用 ==============================
本次token用量：{'input_tokens': 196, 'output_tokens': 12, 'total_tokens': 208, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
============================== 当前是第 2 轮调用 ==============================
本次token用量：{'input_tokens': 196, 'output_tokens': 12, 'total_tokens': 208, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
本次token用量：{'input_tokens': 227, 'output_tokens': 101, 'total_tokens': 328, 'input_token_details': {'cache_read': 191}, 'output_token_details': {}}
============================== 当前是第 3 轮调用 ==============================
本次token用量：{'input_tokens': 196, 'output_tokens': 12, 'total_tokens': 208, 'input_token_details': {'cache_read': 66}, 'output_token_details': {}}
本次token用量：{'input_tokens': 227, 'output_tokens': 101, 'total_tokens': 328, 'input_token_details': {'cache_read': 191}, 'output_token_details': {}}
本次token用量：{'input_tokens': 386, 'output_tokens': 98,